# Detecção de fraude em transações de cartão de crédito

Projeto do curso de Agentes de IA (DIO / Bradesco). Vou seguir o caminho das aulas: coleta, exploração, preparação, modelos (regressão logística, Random Forest e XGBoost), balanceamento, ajuste de limiar, ajuste de hiperparâmetros e explicabilidade com SHAP.

A fraude é muito rara nessa base, então a acurácia engana. O foco vai ser **recall, precisão e F1 da classe de fraude** (classe 1).

In [ ]:
%pip install -q xgboost shap imbalanced-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_curve, roc_auc_score,
                             precision_recall_curve, precision_score, recall_score, f1_score)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap

## 1. Coleta e exploração dos dados

In [ ]:
url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df = pd.read_csv(url)
df.head()

Cada linha é uma transação. `Time` é o tempo desde o início da coleta, `V1` a `V28` são variáveis transformadas por PCA (para proteger a privacidade, por isso não sei o significado de cada uma), `Amount` é o valor e `Class` diz se é fraude (1) ou normal (0).

In [ ]:
print(df.shape)
df_class = df["Class"].value_counts(normalize=True)
df_class

In [ ]:
df["Class"].value_counts().plot(kind="bar", color=["steelblue", "indianred"])
plt.title("Transações por classe (0 = normal, 1 = fraude)")
plt.xticks(rotation=0)
plt.show()

**O que percebi:** quase tudo é normal (cerca de 99,8%) e só cerca de 0,17% é fraude. Um modelo que sempre responde "não é fraude" acertaria quase 100% e não serviria para nada. É um problema de **classificação desbalanceada**, então a acurácia sozinha não serve e vou olhar o recall e a precisão da classe de fraude.

## 2. Feature engineering e preparação

- `np.log1p` em `Amount` (cria `amount_log`): comprime valores muito grandes e o `1p` evita problema com valores iguais a zero.
- `StandardScaler` (cria `amount_scaled`): deixa média 0 e desvio padrão 1.
- Separo `X` (features) e `y` (alvo `Class`).
- `train_test_split` com `test_size=0.3`, `random_state=42` e `stratify=y`, para manter a proporção de fraudes no treino e no teste.

In [ ]:
df["amount_log"] = np.log1p(df["Amount"])

scaler = StandardScaler()
df["amount_scaled"] = scaler.fit_transform(df[["amount_log"]])

# uso amount_scaled no lugar de Amount e amount_log
X = df.drop(["Class", "Amount", "amount_log"], axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("fraudes no treino:", round(y_train.mean(), 5))
print("fraudes no teste :", round(y_test.mean(), 5))

In [ ]:
# guardo as métricas da classe de fraude de cada modelo para comparar no final
resultados = {}


def registrar(nome, y_true, pred):
    resultados[nome] = {
        "precisão": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
    }

## 3. Modelo baseline: Regressão Logística

É um modelo simples de classificação, usado como referência para os outros. Uso `max_iter=1000` para dar mais iterações para o algoritmo convergir. Pode aparecer um *warning* de convergência: não é erro, só indica que o modelo não terminou de aprender no número de iterações (ajudaria melhorar a escala dos dados).

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
pred_lr = log_reg.predict(X_test)

print(classification_report(y_test, pred_lr, target_names=["normal", "fraude"]))
registrar("Regressão Logística", y_test, pred_lr)

A acurácia fica perto de 100%, mas isso é enganoso. O que importa é a linha da classe de fraude: o **recall** mostra quantas fraudes reais o modelo encontrou, e o que falta passou despercebido.

### Curva ROC e curva Precisão × Recall
- **ROC:** mostra recall (taxa de verdadeiros positivos) contra taxa de falsos positivos em vários limiares. Quanto mais perto do canto superior esquerdo, melhor. A diagonal é um modelo aleatório. A AUC perto de 1 é muito bom e perto de 0,5 é aleatório.
- **Precisão × Recall:** mostra o *trade-off*: aumentar o recall costuma reduzir a precisão. É mais informativa que a ROC quando os dados são desbalanceados.

In [ ]:
prob_lr = log_reg.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, prob_lr)
prec, rec, _ = precision_recall_curve(y_test, prob_lr)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, prob_lr):.3f}")
ax[0].plot([0, 1], [0, 1], "k--")
ax[0].set_title("Curva ROC - Regressão Logística")
ax[0].set_xlabel("Taxa de falsos positivos")
ax[0].set_ylabel("Recall (taxa de verdadeiros positivos)")
ax[0].legend()

ax[1].plot(rec, prec)
ax[1].set_title("Curva Precisão x Recall - Regressão Logística")
ax[1].set_xlabel("Recall")
ax[1].set_ylabel("Precisão")
plt.show()

## 4. Técnicas de balanceamento

Duas técnicas principais:

- **Undersampling:** reduz a classe majoritária (normais) até ficar do tamanho da minoritária (fraudes). Equilibra, mas joga fora muitos dados normais.
- **Oversampling:** cria exemplos sintéticos da classe minoritária (uso o **SMOTE**). Não perde dados, mas cria dados artificiais que podem incluir ruído.

Aplico as duas **só no conjunto de treino** e avalio no mesmo teste original, para os resultados serem comparáveis. Para saber qual é melhor, tem que testar.

In [ ]:
# Undersampling: todas as fraudes + a mesma quantidade de transações normais
treino = X_train.copy()
treino["Class"] = y_train

fraudes = treino[treino["Class"] == 1]
normais = treino[treino["Class"] == 0].sample(len(fraudes), random_state=42)
treino_under = pd.concat([fraudes, normais])

X_under = treino_under.drop("Class", axis=1)
y_under = treino_under["Class"]
print("undersampling:", y_under.value_counts().to_dict())

lr_under = LogisticRegression(max_iter=1000)
lr_under.fit(X_under, y_under)
pred_under = lr_under.predict(X_test)
registrar("Reg. Logística + undersampling", y_test, pred_under)
print(classification_report(y_test, pred_under, target_names=["normal", "fraude"]))

In [ ]:
# Oversampling com SMOTE
X_over, y_over = SMOTE(random_state=42).fit_resample(X_train, y_train)
print("oversampling:", y_over.value_counts().to_dict())

lr_over = LogisticRegression(max_iter=1000)
lr_over.fit(X_over, y_over)
pred_over = lr_over.predict(X_test)
registrar("Reg. Logística + SMOTE", y_test, pred_over)
print(classification_report(y_test, pred_over, target_names=["normal", "fraude"]))

## 5. Random Forest

É um conjunto de várias árvores de decisão, e o resultado combina as decisões de todas. Parâmetros que usei:
- `n_estimators=50`: 50 árvores;
- `max_depth=10`: limita a profundidade para evitar *overfitting* (o modelo decorar o treino);
- `class_weight="balanced"`: dá mais peso à classe de fraude, para o modelo não ignorar a classe rara.

In [ ]:
rf = RandomForestClassifier(n_estimators=50, max_depth=10, class_weight="balanced",
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print(classification_report(y_test, pred_rf, target_names=["normal", "fraude"]))
registrar("Random Forest", y_test, pred_rf)

## 6. Pipeline e ajuste de limiar (threshold)

- **Pipeline:** junta `StandardScaler` e regressão logística em um único fluxo. Os dados passam pela padronização e depois pelo modelo, o que evita erros e garante consistência.
- **Threshold:** por padrão o modelo marca fraude se a probabilidade for maior que 0,5. Baixando o limiar, o modelo fica mais sensível: tende a aumentar o recall (mais fraudes detectadas), mas pode gerar mais falsos positivos. Nas aulas foi usado 0,3. Também testo 0,5 e 0,1 para comparar.

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)
prob_pipe = pipe.predict_proba(X_test)[:, 1]

for t in [0.5, 0.3, 0.1]:
    pred_t = (prob_pipe >= t).astype(int)
    registrar(f"Pipeline LR (limiar {t})", y_test, pred_t)

print("Limiar 0.3:")
print(classification_report(y_test, (prob_pipe >= 0.3).astype(int), target_names=["normal", "fraude"]))

## 7. XGBoost

Algoritmo baseado em *boosting*: vários modelos simples (árvores) são treinados **em sequência** e cada um tenta corrigir os erros do anterior. Diferente do Random Forest, em que as árvores são independentes.

In [ ]:
xgb = XGBClassifier(n_estimators=100, max_depth=5, random_state=42,
                    eval_metric="logloss", n_jobs=-1)
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)
prob_xgb = xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred_xgb, target_names=["normal", "fraude"]))
registrar("XGBoost", y_test, pred_xgb)

In [ ]:
# Comparando as curvas dos três modelos principais
probas = {
    "Regressão Logística": prob_lr,
    "Random Forest": rf.predict_proba(X_test)[:, 1],
    "XGBoost": prob_xgb,
}

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for nome, p in probas.items():
    f, t_, _ = roc_curve(y_test, p)
    ax[0].plot(f, t_, label=f"{nome} (AUC={roc_auc_score(y_test, p):.3f})")
    pr, rc, _ = precision_recall_curve(y_test, p)
    ax[1].plot(rc, pr, label=nome)

ax[0].plot([0, 1], [0, 1], "k--")
ax[0].set_title("Curva ROC")
ax[0].set_xlabel("Taxa de falsos positivos")
ax[0].set_ylabel("Recall")
ax[0].legend()
ax[1].set_title("Curva Precisão x Recall")
ax[1].set_xlabel("Recall")
ax[1].set_ylabel("Precisão")
ax[1].legend()
plt.show()

## 8. Importância das variáveis

O gráfico mostra o quanto cada variável contribui para as decisões do modelo: quanto maior a barra, mais ela influencia a previsão. Ajuda a interpretar o modelo, a selecionar variáveis e a entender o negócio.

In [ ]:
importancia = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)

importancia.plot(kind="bar", figsize=(12, 4))
plt.title("Importância das variáveis - XGBoost")
plt.ylabel("Importância")
plt.show()

importancia.head(5)

## 9. Ajuste de hiperparâmetros (GridSearchCV)

Hiperparâmetros (como `max_depth` e `n_estimators`) não são aprendidos pelo modelo, eu é que defino. O `GridSearchCV` testa todas as combinações do `param_grid` e escolhe a melhor pela métrica indicada. Aqui a métrica é o **recall**, porque o objetivo é encontrar o máximo de fraudes.

In [ ]:
param_grid = {
    "max_depth": [3, 5],
    "n_estimators": [50, 100],
}

grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=-1),
    param_grid, scoring="recall", cv=3,
)
grid.fit(X_train, y_train)

print("Melhores parâmetros:", grid.best_params_)
print("Melhor recall (validação cruzada):", round(grid.best_score_, 4))

xgb_grid = grid.best_estimator_
pred_grid = xgb_grid.predict(X_test)
registrar("XGBoost + GridSearch", y_test, pred_grid)
print(classification_report(y_test, pred_grid, target_names=["normal", "fraude"]))

## 10. Explicabilidade com SHAP

Enquanto a importância das variáveis mostra o que é relevante no geral, o SHAP mostra o **impacto de cada variável na previsão**. O `mean(|SHAP|)` é o impacto médio de cada variável nas decisões do modelo: quanto maior, mais ela influencia. Em fraude isso importa para justificar por que uma transação foi marcada, por confiança, auditoria e uso em produção.

In [ ]:
# amostra do teste (todas as fraudes + 1000 normais) para o SHAP rodar rápido
amostra = pd.concat([X_test[y_test == 1], X_test[y_test == 0].sample(1000, random_state=42)])

explainer = shap.TreeExplainer(xgb_grid)
sv = explainer(amostra)

shap.plots.bar(sv, max_display=15)

In [ ]:
shap.plots.beeswarm(sv, max_display=15)

## 11. Comparação final dos modelos (classe de fraude, no teste)

In [ ]:
tabela = pd.DataFrame(resultados).T
tabela.sort_values("recall", ascending=False).round(3)

In [ ]:
# resumo gerado a partir dos resultados desta execução
melhor_recall = tabela["recall"].idxmax()
melhor_prec = tabela["precisão"].idxmax()
melhor_f1 = tabela["F1"].idxmax()
top_shap = pd.Series(np.abs(sv.values).mean(axis=0), index=amostra.columns).sort_values(ascending=False)

print(f"Maior recall   : {melhor_recall} ({tabela.loc[melhor_recall, 'recall']:.2f})")
print(f"Maior precisão : {melhor_prec} ({tabela.loc[melhor_prec, 'precisão']:.2f})")
print(f"Maior F1       : {melhor_f1} ({tabela.loc[melhor_f1, 'F1']:.2f})")
print("Melhores parâmetros do GridSearch:", grid.best_params_)
print("Variáveis mais influentes no SHAP:", ", ".join(top_shap.head(3).index))

- **Acurácia engana:** todos os modelos ficaram com acurácia perto de 1,00. A diferença entre eles aparece só no recall, na precisão e no F1 da classe de fraude.
- **Melhor modelo:** o Random Forest teve o melhor F1 (0,799), com recall de 0,764 e precisão de 0,837, e a maior AUC (0,968). Foi o modelo mais equilibrado.
- **Trade-off:** o undersampling (recall 0,878) e o SMOTE (recall 0,851) encontraram mais fraudes, mas a precisão caiu para 0,056 e 0,141. Na prática, quase todos os alertas seriam falsos alarmes. O mesmo trade-off apareceu ao baixar o limiar da regressão logística.
- **Balanceamento:** neste teste, usar pesos de classe (como no Random Forest) funcionou melhor do que equilibrar os dados. Isso vale para esta base e esta divisão, e a única forma de saber é testar.
- **Explicabilidade:** poucas variáveis (V16, V11, V10) concentram a decisão do modelo, o que ajuda a justificar por que uma transação foi marcada como fraude.